# 15_04 — Evaluación predictiva · Escenario 5

Paso **3b** del ciclo. Supone que `15_03_convergencia` dio veredicto correcto;
si las cadenas no convergieron, nada de lo que sigue significa algo.

## Qué se reporta

| Sección | Qué responde |
|---|---|
| 4 | métricas puntuales y distribucionales, **train vs test** |
| 5 | bandas de credibilidad sobre la serie de cada score |
| 6 | intervalos sobre la curva, en extractos cada 10 períodos |
| 7 | ventana móvil: cómo evoluciona el error al cruzar $T_0$ |
| 8 | calibración marginal: PIT por bloque |
| **9** | **diagnóstico de la representación: dónde está la señal y dónde se trunca** |
| 10 | comparación con las líneas base |

## Cómo leer este escenario — y qué NO concluir de él

Éste es un escenario del **Bloque 2**, y su lectura difiere de la de los cuatro
primeros en un punto que hay que declarar antes de mirar una sola cifra:

> **El resultado no discrimina entre especificaciones dinámicas.** Si el
> truncamiento descarta la dirección informativa, la degradación alcanza por
> igual al PSBPM-FD, al FAR(1), al VAR sobre scores y a cualquier otro método
> sobre la misma representación. Lo que este escenario mide es el **alcance de
> la reducción de dimensión**, no la calidad relativa de los modelos.

De ahí se siguen tres cosas:

1. **Un $R^2$ bajo aquí NO es un mal resultado del modelo.** Es la medición. Si
   las componentes retenidas son ruido blanco por construcción, el $R^2$ óptimo
   alcanzable es cero y la media incondicional es la predicción correcta. Leerlo
   como fracaso del PSBPM-FD sería un error.
2. **Ganarle a las líneas base tampoco es exigible.** Cuando la señal está fuera
   del truncamiento, todos los métodos empatan, y el empate es el resultado.
3. **§9 es la sección que da sentido a las demás.** Muestra, componente por
   componente, dónde está la varianza y dónde está la predictibilidad, y si la
   dirección informativa quedó dentro o fuera. Sin esa sección, las cifras de §4
   no se pueden interpretar.

**Y el aviso principal de esta corrida**: con el $M$ del invariante del estudio
el truncamiento **retiene** la dirección informativa, de modo que la corrida es
un **caso nulo** —válida y reportable como control, pero no la prueba que
`§03_06` pide—. `15_01 §3.4` lo declara en `eval_config.json`, y este notebook
lo verifica y lo repite en §9.

## 1. Imports, rutas y artefactos

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

from model_psbp_fd.pipelines import (
    cargar_curvas, cargar_curvas_true, cargar_fpca, cargar_estandarizador,
    cargar_datasets_ar, cargar_hiperparametros, cargar_config_evaluacion,
    cargar_escenario,
)
from model_psbp_fd.functions_models import DataStandardizer
from model_psbp_fd.models.pspb_fd_v3 import PSBPPredictor, PropagadorFuncional

from model_psbp_fd.fit import (
    rmse_por_coeficiente, r2_por_columna, razon_dispersion, mise, rmse_funcional,
    crps_muestral, energy_score, cobertura, intervalo_muestral,
    pit_muestral, diagnostico_pit,
    agrupar_momentos, ventana_movil_scores, ventana_movil_funcional,
)
from model_psbp_fd.graphics import (
    plot_ventana_movil, plot_bandas_serie, plot_extractos_curvas,
    plot_calibracion_pit, plot_scatter_theta,
)
from model_psbp_fd.utils import get_project_root
from model_psbp_fd.utils.quadrature import pesos_trapezoidales

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

In [ ]:
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# [CONFIG] debe coincidir con 15_01, 15_03 y psbp_fd_iteracion.m
BASENAME, ESCENARIO_ID, REPLICA_ID = "escenario", 5, 1
EXPERIMENT_ID = f"{BASENAME}_{ESCENARIO_ID}_r{REPLICA_ID:02d}"

PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / EXPERIMENT_ID,
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / EXPERIMENT_ID,
    "out_report":   PROJECT_ROOT / "reports" / "simulaciones" / EXPERIMENT_ID,
    "out_artefact": PROJECT_ROOT / "artefact" / "simulaciones" / EXPERIMENT_ID,
}
for r in PATHS.values():
    r.mkdir(parents=True, exist_ok=True)
print(f"EXPERIMENT_ID : {EXPERIMENT_ID}")

In [ ]:
dfs_train, manifest = cargar_datasets_ar(PATHS, bloque="train")
dfs_test,  _        = cargar_datasets_ar(PATHS, bloque="test")
hp_json     = cargar_hiperparametros(PATHS)
eval_config = cargar_config_evaluacion(PATHS)

COMPONENT_IDX = manifest["component_idx"]
n_components  = len(COMPONENT_IDX)
cov_names     = manifest["cov_names"]
N_LAGS        = int(manifest["n_lags"])
T, T0         = int(manifest["T"]), int(manifest["T0"])
N_ITER        = int(hp_json["n_iter"])
MCMC_CFG      = hp_json["mcmc_config"]
BURN          = int(MCMC_CFG["burn"])

assert manifest["scores_scale"] == "standardized_zscore_ddof0"

# Parámetros de evaluación: se LEEN del artefacto, no se redeclaran aquí.
NIVEL        = float(eval_config.get("nivel_credibilidad", 0.95))
MODO_RESIDUO = eval_config.get("modo_residuo", "ninguno")
OBJETIVO     = eval_config.get("objetivo_evaluacion", "curva_verdadera")
VENTANAS_W   = list(eval_config.get("ventana_movil", {}).get("w", [10, 20, 40]))
REPR_CFG     = eval_config.get("representacion", {})

assert eval_config.get("estratificacion") is None, (
    "Este notebook está escrito para un escenario SIN estado latente. "
    "eval_config declara una estratificación: revisar 15_01.")

print(f"T={T}  T0={T0}  n_lags={N_LAGS}  componentes={n_components}")
print(f"nivel={NIVEL}  ·  modo_residuo={MODO_RESIDUO!r}  ·  objetivo={OBJETIVO!r}")
print(f"ventanas w = {VENTANAS_W}")
print(f"\nestratificación: NINGUNA. Motivo declarado en el artefacto:")
print("  " + eval_config.get("estratificacion_motivo", "(no declarado)"))

RETIENE   = bool(REPR_CFG.get("retiene_direccion_informativa", True))
CASO_NULO = bool(REPR_CFG.get("es_caso_nulo", RETIENE))
print("\n" + "=" * 70)
print("ADVERTENCIA DE LECTURA (declarada en eval_config.json):")
print("  " + REPR_CFG.get("advertencia", ""))
print(f"\n  M retenidas = {REPR_CFG.get('M_retenidas')}   ·   la dinámica vive "
      f"en FPC {REPR_CFG.get('fpc_con_dinamica')}")
print(f"  ¿el truncamiento la retiene? {'SÍ' if RETIENE else 'NO'}   "
      f"·   ¿caso nulo? {'SÍ' if CASO_NULO else 'NO'}")
print("=" * 70)

In [ ]:
fpca = cargar_fpca(PATHS)
_ver = fpca.verificar()
assert _ver["todo_ok"], f"Las identidades FPCA no se cumplen: {_ver}"

scores_standardizer = cargar_estandarizador(PATHS, DataStandardizer)
X_obs, grilla = cargar_curvas(PATHS)                 # (T, G) con ruido
X_true        = cargar_curvas_true(PATHS)            # (T, G) verdadera — el objetivo
w_quad        = pesos_trapezoidales(grilla)

M_fpca   = fpca.M
Psi_grid, mu_grid = fpca.Psi_grid, fpca.mu_grid
SCORES     = fpca.SCORES                              # (T, M) escala original
SCORES_STD = scores_standardizer.transform(SCORES)

print(f"FPCA M={M_fpca} K={fpca.K}   ·   curvas {X_true.shape}  grilla {grilla.shape}")
print(f"sd(observada - verdadera) = {(X_obs - X_true).std():.4f}   "
      "← el ruido que el modelo NO debe predecir")
assert COMPONENT_IDX == list(range(M_fpca)), (
    "La propagación funcional necesita el vector completo de scores en orden; "
    f"COMPONENT_IDX={COMPONENT_IDX} y M={M_fpca}.")

### 1.1 La estructura verdadera del generador

En lugar del estado latente que leen las corridas 12 y 13, aquí se lee la
**estructura de la representación**: el espectro del generador, los $\varphi_j$
y la alineación entre las componentes FPCA estimadas y las de Fourier. `15_01`
las calculó y persistió; ninguna entra en predicción alguna.

Los coeficientes verdaderos $a_{tj}$ se leen del `.npz` crudo. Con ellos §9
puede medir directamente cuánta señal quedó fuera del truncamiento, que es la
cantidad que este escenario existe para producir.

In [ ]:
_csv_alin = PATHS["out_report"] / "10_alineacion_fpca_generador.csv"
_csv_estr = PATHS["out_report"] / "10_estructura_generador.csv"
assert _csv_alin.exists() and _csv_estr.exists(), (
    "Faltan las tablas de estructura que escribe 15_01 (§2.3 y §4). "
    "Regenerar 15_01.")
alineacion_df  = pd.read_csv(_csv_alin)
estructura_df  = pd.read_csv(_csv_estr)

_crudo = cargar_escenario(str(PATHS["raw"] / f"escenario_{ESCENARIO_ID}.npz"))
assert "interno_coeficientes" in _crudo, (
    "El .npz no contiene `interno_coeficientes`: regenerar 15_01 con "
    "guardar_escenario(..., incluir_internos=True).")
A_coef = _crudo["interno_coeficientes"][REPLICA_ID - 1]      # (T, J) verdaderos
PHIS   = _crudo["interno_coeficientes_ar"]                   # (J,)
J_GEN  = A_coef.shape[1]
J_ESTRELLA = int(REPR_CFG.get("indice_predecible_fourier",
                              int(np.argmax(np.abs(PHIS))) + 1))

# Verificación numérica contra la fuente, no contra el CSV.
assert np.allclose(PHIS, np.asarray(REPR_CFG["phis_generador"], float)), \
    "Los phi del .npz no coinciden con los declarados en eval_config.json."
assert PHIS[J_ESTRELLA - 1] != 0.0 and np.count_nonzero(PHIS) == 1, \
    "El generador no tiene exactamente una componente con dinámica."

print(f"coeficientes verdaderos {A_coef.shape}  ·  J={J_GEN}")
print(f"componente de Fourier con dinámica: j*={J_ESTRELLA} "
      f"(phi={PHIS[J_ESTRELLA-1]:+.2f}); todas las demás phi=0 exacto")
display(alineacion_df.head(min(10, len(alineacion_df))).style.format({
    "corr_alineacion": "{:.4f}", "var_ratio": "{:.4%}",
    "ar1_score": "{:+.4f}", "phi_generador": "{:+.2f}"}))

## 2. Trazas MCMC

In [ ]:
def ruta_traza(fpc_idx, chain):
    return PATHS["out_artefact"] / f"chain_fpc_{fpc_idx}_iter{chain:02d}.mat"


def leer_traza(path):
    m = loadmat(str(path))
    claves = ["betajhout", "beta0hout", "tauhout", "alphahout", "psijhout",
              "Gammajhout", "gammajhout", "pijout", "wjout", "osumout", "inEout"]
    traces = {k: np.asarray(m[k], dtype=np.float64) for k in claves}
    burn = int(np.asarray(m["burn"]).ravel()[0])
    feat = str(np.atleast_1d(m["feature_names"]).ravel()[0]).split(",")
    return traces, burn, feat


class ModeloTraza:
    def __init__(self, traces, burn, feature_names):
        self.traces = traces
        self.feature_names_ = list(feature_names)
        self.burn = int(burn)
        self.predictor_ = PSBPPredictor(traces=traces, burn=burn)

    def _diseno(self, df):
        Xp = np.asarray(df.iloc[:, 1:], dtype=float)
        return np.hstack([np.ones((Xp.shape[0], 1)), Xp])

    def momentos(self, df):
        return self.predictor_.momentos_predictivos(self._diseno(df))

    def muestrear(self, df, d=1, seed=None):
        return self.predictor_.muestrear(self._diseno(df), d, seed=seed)


faltan = [ruta_traza(COMPONENT_IDX[k] + 1, c + 1).name
          for k in range(n_components) for c in range(N_ITER)
          if not ruta_traza(COMPONENT_IDX[k] + 1, c + 1).exists()]
assert not faltan, f"Faltan trazas: {faltan}. Ejecuta psbp_fd_iteracion.m."

models_chains = {k: {} for k in range(n_components)}
for k in range(n_components):
    esperado = list(dfs_train[k].columns[1:])
    for c in range(N_ITER):
        traces, burn, feat = leer_traza(ruta_traza(COMPONENT_IDX[k] + 1, c + 1))
        assert feat == esperado, f"[k={k} c={c+1}] feature_names != columnas del dataset."
        models_chains[k][c] = ModeloTraza(traces, burn, feat)

n_post = MCMC_CFG["nsim"] - BURN
print(f"OK  {n_components} x {N_ITER} cadenas · {n_post} draws posteriores c/u")

## 3. Predicción a $h=1$ sobre la serie completa

Los dos bloques se concatenan en **una sola serie de orígenes**
$t=N_{lags}+1,\dots,T$. En todos ellos la predicción usa los **rezagos
reales** —nunca predicciones encadenadas—, de modo que el horizonte es 1 en todo
el recorrido y el modelo no se reentrena en ningún punto.

In [ ]:
S_POR_ITER = 100
SEED_PRED  = 20260823

dfs_full = {k: pd.concat([dfs_train[k], dfs_test[k]], ignore_index=True)
            for k in range(n_components)}
n_orig   = len(dfs_full[0])
t_orig   = np.arange(N_LAGS + 1, T + 1)          # tiempo del experimento, base-1
assert len(t_orig) == n_orig, f"{len(t_orig)} != {n_orig}"

T0_orig  = T0 - N_LAGS
es_train = t_orig <= T0

print(f"orígenes evaluados: {n_orig}  (train {es_train.sum()} · test {(~es_train).sum()})")

In [ ]:
Y_obs  = np.column_stack([dfs_full[k].iloc[:, 0].to_numpy() for k in range(n_components)])
Y_hat  = np.empty_like(Y_obs)
Y_sd   = np.empty_like(Y_obs)
draws  = []          # por componente: (S, n_orig)

for k in range(n_components):
    medias, sds, muestras_k = [], [], []
    for c in sorted(models_chains[k]):
        mom = models_chains[k][c].momentos(dfs_full[k])
        medias.append(mom["media"])
        sds.append(mom["sd"])            # PREDICTIVA (v3), no la del centro
        muestras_k.append(models_chains[k][c].muestrear(
            dfs_full[k], S_POR_ITER, seed=SEED_PRED + 1000 * k + c))
    Y_hat[:, k], Y_sd[:, k] = agrupar_momentos(np.column_stack(medias),
                                               np.column_stack(sds))
    draws.append(np.concatenate(muestras_k, axis=0))

SC_draws = np.stack(draws, axis=2)                  # (S, n_orig, M)
S_total  = SC_draws.shape[0]
li_s, ls_s = intervalo_muestral(SC_draws, nivel=NIVEL)   # (n_orig, M)

print(f"extracciones por score: S = {S_total} "
      f"= {n_post} draws x {S_POR_ITER} x {N_ITER} cadenas")
print(f"SC_draws {SC_draws.shape}  ·  bandas {li_s.shape}")

### 3.1 Propagación a la curva

$\hat X^{(s)}_t(\tau)=\mu(\tau)+\sum_m (d_m\tilde\xi^{(s)}_{tm}+c_m)\psi_m(\tau)$.
Con `modo_residuo="ninguno"` el mapa es determinista y las curvas son
extracciones exactas de la predictiva de la curva **proyectada**.

Aquí conviene distinguir **dos pisos** de error, y este escenario es el único
donde la distinción importa:

- el **truncamiento FPCA**, que es el objeto de estudio;
- la **base B-spline**, que introduce error adicional porque el proceso vive en
  el span de $J=10$ funciones de Fourier y el estudio representa con B-splines.

`15_01 §3.2` midió el segundo y lo dejó en `eval_config`; abajo se separan.

In [ ]:
S_FUNC = 500
paso_thin = max(1, S_total // S_FUNC)
SC_thin = SC_draws[::paso_thin]

propagador = PropagadorFuncional(Psi_grid, mu_grid,
                                 estandarizador=scores_standardizer,
                                 modo_residuo=MODO_RESIDUO)
X_draws = propagador.curvas_desde_scores(SC_thin, seed=SEED_PRED)   # (S', n, G)
X_pred  = X_draws.mean(axis=0)                                       # (n, G)
li_f, ls_f = np.quantile(X_draws, (1 - NIVEL) / 2, axis=0), \
             np.quantile(X_draws, 1 - (1 - NIVEL) / 2, axis=0)

X_true_ev = X_true[N_LAGS:]     # (n_orig, G) VERDADERA — contra esto se evalúa
X_obs_ev  = X_obs[N_LAGS:]      # (n_orig, G) observada, sólo para las figuras

print(f"muestras funcionales {X_draws.shape}  (adelgazado 1 de cada {paso_thin})")
print(f"memoria aprox {X_draws.nbytes / 1e6:.0f} MB")

X_proj = fpca.reconstruct(SCORES)[N_LAGS:]
_mise_trunc = mise(X_true_ev, X_proj, grilla)
_mise_bspl  = float(REPR_CFG.get("mise_bspline_vs_verdadera", np.nan))
print(f"\nMISE del TRUNCAMIENTO (proyección FPCA + B-spline): {_mise_trunc:.6f}")
print(f"  de los cuales, atribuible a la base B-spline    : {_mise_bspl:.6f}")
print(f"  atribuible al truncamiento FPCA (la diferencia) : "
      f"{_mise_trunc - _mise_bspl:.6f}")
print("   ← ningún modelo sobre esta representación puede bajar del primero")

## 4. Métricas, entrenamiento contra prueba

**Antes de leer la tabla, releer la advertencia de la cabecera.** En este
escenario un $R^2$ bajo no mide la calidad del modelo sino cuánta señal
sobrevivió al truncamiento. La cifra sólo se interpreta junto a §9.

Lo que sí conserva su significado habitual es la comparación train/test: el
proceso es estacionario, de modo que cualquier degradación fuera de muestra es
generalización y no cambio del mecanismo. Es el contraste con el Algoritmo 6,
donde sí cambia.

In [ ]:
def _metricas_bloque(mask, etiqueta):
    filas = []
    for k in range(n_components):
        y, p = Y_obs[mask, k], Y_hat[mask, k]
        z    = SC_draws[:, mask, k]
        cob  = cobertura(y, li_s[mask, k], ls_s[mask, k])
        pit  = pit_muestral(y, z)
        filas.append({
            "bloque":  etiqueta,
            "FPC":     f"FPC {COMPONENT_IDX[k] + 1}",
            "RMSE":    float(rmse_por_coeficiente(y[:, None], p[:, None])[0]),
            "R2":      float(r2_por_columna(y[:, None], p[:, None], centrar=True)[0]),
            "sd_ratio": float(razon_dispersion(y[:, None], p[:, None])[0]),
            "CRPS":    float(crps_muestral(y, z).mean()),
            f"Cob{int(NIVEL*100)}": cob["cobertura"],
            "Ancho":   cob["ancho_medio"],
            "PIT_ks":  float(diagnostico_pit(pit)["ks"]),
            "PIT_forma": diagnostico_pit(pit)["forma"],
            "n":       int(mask.sum()),
        })
    return filas

met_df = pd.DataFrame(_metricas_bloque(es_train, "train")
                      + _metricas_bloque(~es_train, "test"))
met_df = met_df.set_index(["bloque", "FPC"])
met_df.to_csv(PATHS["out_report"] / "50_metricas_scores.csv")

_num = [c for c in met_df.columns if met_df[c].dtype.kind == "f"]
display(met_df.style
    .format({c: "{:.4f}" for c in _num})
    .background_gradient(subset=["RMSE", "CRPS"], cmap="RdYlGn_r")
    .background_gradient(subset=["R2"], cmap="RdYlGn", vmin=0, vmax=1)
    .background_gradient(subset=[f"Cob{int(NIVEL*100)}"], cmap="RdYlGn",
                         vmin=0.80, vmax=1.0)
    .set_caption("Métricas por score y bloque · CRPS y cobertura desde la "
                 "predictiva MUESTRAL"))

_deg = (met_df.loc["test", "RMSE"].mean() / met_df.loc["train", "RMSE"].mean())
print(f"\nRMSE test / RMSE train = {_deg:.3f}"
      + ("   sin degradación apreciable" if _deg < 1.15
         else "   ← el modelo se degrada fuera de muestra"))

# El R2 por componente contra el phi verdadero: la comparación que interpreta
# la tabla. Para un AR(1) con coeficiente phi, el R2 poblacional es phi^2.
print(f"\nR2 alcanzable por componente (AR(1) verdadero ⇒ R2 = phi^2):")
_ret = alineacion_df[alineacion_df["retenida"]].set_index("fpc")
for k in range(n_components):
    fpc  = COMPONENT_IDX[k] + 1
    _phi = float(_ret.loc[fpc, "phi_generador"])
    _r2o = float(met_df.loc[("test", f"FPC {fpc}"), "R2"])
    print(f"  FPC {fpc}: phi={_phi:+.2f}  R2 máximo={_phi**2:.4f}  "
          f"R2 observado={_r2o:+.4f}")
print("   Un R2 observado cercano a cero DONDE phi = 0 es el resultado "
      "correcto, no un\n   fallo: esa componente es ruido blanco por "
      "construcción y nada puede predecirla.")

In [ ]:
# Métricas funcionales, contra la curva VERDADERA
filas_f = []
for mask, etq in ((es_train, "train"), (~es_train, "test")):
    dentro = (X_true_ev[mask] >= li_f[mask]) & (X_true_ev[mask] <= ls_f[mask])
    filas_f.append({
        "bloque": etq,
        "MISE":   mise(X_true_ev[mask], X_pred[mask], grilla),
        "RMSE_f": rmse_funcional(X_true_ev[mask], X_pred[mask], grilla),
        "MISE_truncamiento": mise(X_true_ev[mask], X_proj[mask], grilla),
        "energy": energy_score(X_true_ev[mask], X_draws[:, mask, :],
                               max_pares=2000, seed=0),
        f"Cob{int(NIVEL*100)}_puntual": float(dentro.mean()),
        "ancho_medio": float((ls_f[mask] - li_f[mask]).mean()),
        "n": int(mask.sum()),
    })
fun_df = pd.DataFrame(filas_f).set_index("bloque")
fun_df.to_csv(PATHS["out_report"] / "51_metricas_funcionales.csv")

display(fun_df.style.format("{:.6f}", subset=["MISE", "RMSE_f", "MISE_truncamiento"])
        .format("{:.4f}", subset=["energy", f"Cob{int(NIVEL*100)}_puntual", "ancho_medio"])
        .set_caption("Métricas funcionales contra la curva VERDADERA · "
                     "banda sin residuo de representación"))

_frac = fun_df["MISE_truncamiento"] / fun_df["MISE"]
print("\nfracción del MISE atribuible al truncamiento de la representación:")
for b in fun_df.index:
    print(f"  {b:5s}: {_frac[b]:.1%}")
print("\nEn el Bloque 2 esta fracción ES el resultado. Si domina, el techo de la "
      "\nrepresentación manda y mejorar el modelo no rinde: sólo subir M —o "
      "cambiar la\nbase— puede mover la aguja.")
print("La cobertura es PUNTUAL (cada tau por separado), NO simultánea.")

## 5. Bandas de credibilidad sobre la serie de scores

In [ ]:
plot_bandas_serie(
    Y_obs, Y_hat, li_s, ls_s, T0, t=t_orig,
    etiquetas=[f"FPC {i + 1}" for i in COMPONENT_IDX], nivel=NIVEL,
    title="Bandas de credibilidad por score — entrenamiento y prueba",
    save_path=str(PATHS["out_report"] / "52_bandas_scores.png"))
plt.show()
print("Lectura del escenario: en las componentes con phi = 0 el CENTRO debe ser "
      "plano —la\nmedia incondicional— y la banda ancha. Sólo en la componente "
      "con dinámica el\ncentro debe seguir a la serie. Un centro que oscile "
      "donde phi = 0 es\nsobreajuste al ruido.")

# Cuánto se mueve el centro predicho en cada componente, contra cuánto DEBERÍA.
print("\nsd del centro predicho / sd de la respuesta, por componente:")
_ret = alineacion_df[alineacion_df["retenida"]].set_index("fpc")
for k in range(n_components):
    fpc  = COMPONENT_IDX[k] + 1
    _phi = float(_ret.loc[fpc, "phi_generador"])
    _r   = float(Y_hat[:, k].std() / max(Y_obs[:, k].std(), 1e-12))
    print(f"  FPC {fpc}: {_r:.4f}   (valor correcto |phi| = {abs(_phi):.2f})")

In [ ]:
eval_scatter = {k: {"y_obs": Y_obs[~es_train, k], "y_hat": Y_hat[~es_train, k]}
                for k in range(n_components)}
plot_scatter_theta(eval_scatter, n_components=n_components,
                   save_path=str(PATHS["out_report"] / "53_scatter_scores_test.png"))
plt.show()

## 6. Intervalos de credibilidad sobre la curva

Extractos de la serie funcional cada `CADA` períodos, cada uno con su banda
puntual, la curva verdadera y —en gris— los datos observados.

In [ ]:
CADA = 10   # [CONFIG] un extracto cada CADA períodos

plot_extractos_curvas(
    X_true_ev, X_pred, li_f, ls_f, grilla, T0, t=t_orig,
    cada=CADA, n_col=5, nivel=NIVEL, X_obs=X_obs_ev,
    title="Predictiva funcional y banda de credibilidad",
    save_path=str(PATHS["out_report"] / "54_extractos_curvas.png"))
plt.show()

In [ ]:
# Zoom sobre la frontera train/test
i_corte = int(np.searchsorted(t_orig, T0))
sel = np.arange(max(0, i_corte - 3), min(n_orig, i_corte + 4))

plot_extractos_curvas(
    X_true_ev[sel], X_pred[sel], li_f[sel], ls_f[sel], grilla, T0,
    t=t_orig[sel], cada=1, n_col=len(sel), nivel=NIVEL, X_obs=X_obs_ev[sel],
    title=f"Frontera train/test (T0={T0})",
    save_path=str(PATHS["out_report"] / "55_extractos_frontera.png"))
plt.show()
print("En un proceso estacionario la frontera no debe mostrar nada especial. "
      "Que no lo\nmuestre es el contraste con el Algoritmo 6, donde sí lo hace.")

## 7. Ventana móvil

El modelo **no se reentrena**: lo que se desliza es la ventana de evaluación.
Las ventanas que **cruzan** $T_0$ van punteadas, porque su cifra mezcla dentro y
fuera de muestra.

Lectura propia del escenario: el proceso es estacionario y homogéneo, de modo
que **no debe haber ni tendencia ni salto en $T_0$**. Una serie plana con ruido
es aquí el resultado correcto, y sirve de referencia visual para la corrida 16,
donde la misma figura debe romperse en el quiebre.

In [ ]:
tablas_score = {
    w: ventana_movil_scores(
        Y_obs, Y_hat, T0_orig, w=w, muestras=SC_draws, li=li_s, ls=ls_s,
        t_offset=N_LAGS,
        etiquetas=[f"FPC {i + 1}" for i in COMPONENT_IDX])
    for w in VENTANAS_W
}
w_ref = VENTANAS_W[len(VENTANAS_W) // 2]
tablas_score[w_ref].to_csv(PATHS["out_report"] / f"57_ventana_scores_w{w_ref}.csv",
                           index=False)

plot_ventana_movil(
    tablas_score[w_ref], T0, ["rmse", "r2_local", "crps", "cobertura"],
    columna_grupo="componente",
    title=f"Ventana móvil por score (w={w_ref})",
    save_path=str(PATHS["out_report"] / "57_ventana_scores.png"))
plt.show()

In [ ]:
tablas_fun = {
    w: ventana_movil_funcional(X_true_ev, X_pred, grilla, T0_orig, w=w,
                               li=li_f, ls=ls_f, t_offset=N_LAGS)
    for w in VENTANAS_W
}
tablas_fun[w_ref].to_csv(PATHS["out_report"] / f"58_ventana_funcional_w{w_ref}.csv",
                         index=False)

plot_ventana_movil(
    tablas_fun[w_ref], T0, ["mise", "mise_rel", "cobertura_puntual"],
    tablas_por_w=tablas_fun,
    title="Ventana móvil del error funcional (contra la curva verdadera)",
    save_path=str(PATHS["out_report"] / "58_ventana_funcional.png"))
plt.show()

In [ ]:
# Lectura numérica del salto en T0, excluyendo las ventanas que lo cruzan.
print(f"{'w':>4}  {'MISE train':>11}  {'MISE test':>11}  {'salto':>7}")
print(f"{'-'*4}  {'-'*11}  {'-'*11}  {'-'*7}")
for w in VENTANAS_W:
    t_ = tablas_fun[w]
    limpio = t_[~t_["cruza_T0"]]
    a = limpio.loc[limpio.bloque == "train", "mise"].mean()
    b = limpio.loc[limpio.bloque == "test",  "mise"].mean()
    print(f"{w:>4}  {a:>11.6f}  {b:>11.6f}  {b/a:>6.2f}x")

# Test de tendencia: en un proceso estacionario no debe haberla.
t_ref = tablas_fun[w_ref]
if "t_fin" in t_ref.columns:
    _c = float(np.corrcoef(t_ref["t_fin"].to_numpy(),
                           t_ref["mise"].to_numpy())[0, 1])
    print(f"\ncorr(t, MISE local) = {_c:+.3f}")
    print("   Cercana a cero: el error no tiene deriva, como corresponde a un "
          "proceso\n   estacionario. Es la referencia contra la cual leer la "
          "misma figura del\n   Algoritmo 6, donde el quiebre debe producir un "
          "salto.")

## 8. Calibración marginal

Bajo calibración perfecta el PIT es uniforme. La **forma** dice qué falla: una
U indica bandas demasiado angostas, una campana demasiado anchas, y una
pendiente un sesgo del centro.

En este escenario el PIT debería salir **uniforme y sin drama**: la ley
condicional es gaussiana y está dentro de lo que la mezcla puede representar. Un
PIT bien calibrado aquí es la confirmación de que el modelo no rompe nada cuando
el supuesto gaussiano es correcto — el control que hace creíbles los resultados
de las corridas 12, 13 y 14.

In [ ]:
pit_tr = {f"FPC {COMPONENT_IDX[k]+1}": pit_muestral(Y_obs[es_train, k],
                                                    SC_draws[:, es_train, k])
          for k in range(n_components)}
pit_te = {f"FPC {COMPONENT_IDX[k]+1}": pit_muestral(Y_obs[~es_train, k],
                                                    SC_draws[:, ~es_train, k])
          for k in range(n_components)}

plot_calibracion_pit(pit_tr, pit_te,
                     save_path=str(PATHS["out_report"] / "60_pit.png"))
plt.show()

for nombre in pit_tr:
    d_tr, d_te = diagnostico_pit(pit_tr[nombre]), diagnostico_pit(pit_te[nombre])
    print(f"  {nombre}:  train KS={d_tr['ks']:.3f} ({d_tr['forma']})   "
          f"test KS={d_te['ks']:.3f} ({d_te['forma']})")
print("\nUn PIT uniforme aquí es el CONTROL del estudio: confirma que la mezcla "
      "no\nintroduce mala calibración cuando la condicional verdadera es "
      "gaussiana.")

## 9. Diagnóstico de la representación

**La sección propia de este escenario**, y la que da sentido a todas las
anteriores. Ocupa el lugar que en las corridas 12, 13 y 16 ocupa la calibración
condicional al estado verdadero, y no por sustitución: aquí no hay estado, hay
una **representación** que puede estar dejando fuera la señal.

Se responden tres preguntas, en orden:

1. **¿Dónde está la varianza y dónde la predictibilidad?** La figura de la
   izquierda ordena las componentes por varianza —el criterio de truncamiento—;
   la de la derecha, por su AR(1) propio. El desalineamiento entre ambas es el
   mecanismo del escenario.
2. **¿La dirección informativa quedó dentro o fuera de las $M$ retenidas?** De
   esto depende todo lo demás, y `15_01` ya lo declaró en `eval_config.json`.
3. **¿Cuánta señal quedó fuera?** Se mide directamente sobre los coeficientes
   verdaderos $a_{tj}$ del generador: qué fracción de la varianza predecible del
   proceso vive en las componentes descartadas. Es la cifra que cuantifica el
   costo de la reducción de dimensión, con independencia del modelo que se
   ajuste encima.

In [ ]:
# ── (1) y (2): varianza contra predictibilidad, con el corte marcado ─────────
_ar1_score = alineacion_df["ar1_score"].to_numpy()
_varr      = alineacion_df["var_ratio"].to_numpy()
_ret_mask  = alineacion_df["retenida"].to_numpy()
_pred_mask = alineacion_df["es_predecible"].to_numpy()
K_FPCA     = len(alineacion_df)

_n = min(10, K_FPCA)
_kx = np.arange(1, _n + 1)
_col = ["#c0392b" if _pred_mask[k] else "#2980b9" for k in range(_n)]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].bar(_kx, _varr[:_n], color=_col, alpha=0.85)
axes[0].set_yscale("log"); axes[0].set_xticks(_kx)
axes[0].set_ylabel("var_ratio"); axes[0].set_xlabel("componente FPCA")
axes[0].set_title("Criterio de truncamiento: VARIANZA", fontsize=10)

axes[1].bar(_kx, np.abs(_ar1_score[:_n]), color=_col, alpha=0.85)
axes[1].set_xticks(_kx); axes[1].set_ylim(0, 1)
axes[1].set_ylabel(r"$|ar1|$ del score"); axes[1].set_xlabel("componente FPCA")
axes[1].set_title("Lo que el criterio NO mira: PREDICTIBILIDAD", fontsize=10)

for ax in axes:
    ax.axvline(M_fpca + 0.5, color="k", ls="--", lw=1.6)
    ax.text(M_fpca + 0.55, ax.get_ylim()[1], f" corte $M$={M_fpca}",
            fontsize=9, va="top")
fig.suptitle("Escenario 5 — dónde está la señal y dónde cae el truncamiento "
             "(rojo = componente con dinámica verdadera)", fontsize=12)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "61_diagnostico_representacion.png",
            dpi=150, bbox_inches="tight")
plt.show()

_fpc_din = [int(f) for f in alineacion_df.loc[_pred_mask, "fpc"]]
print(f"componentes FPCA con dinámica verdadera : {_fpc_din}")
print(f"componentes retenidas                   : "
      f"{alineacion_df.loc[_ret_mask, 'fpc'].tolist()}")
print(f"\n¿el truncamiento retiene la dirección informativa? "
      f"{'SÍ' if RETIENE else 'NO'}")

In [ ]:
# ── (3) ¿cuánta señal quedó fuera? Se mide sobre los coeficientes VERDADEROS ─
# Para un AR(1) de coeficiente phi y varianza marginal lambda, la varianza
# PREDECIBLE a un paso es phi^2 * lambda: es lo que un predictor óptimo puede
# explicar. La suma sobre componentes es la señal total disponible en el proceso.
var_A       = A_coef.var(axis=0)                     # (J,) varianza empírica
var_predec  = (PHIS ** 2) * var_A                    # (J,) varianza predecible
señal_total = float(var_predec.sum())

# Qué componentes de Fourier están representadas por las M FPC retenidas.
_fourier_ret = sorted(set(int(f) for f in
                          alineacion_df.loc[_ret_mask, "fourier_alineada"]))
_dentro = np.zeros(J_GEN, bool)
for j in _fourier_ret:
    _dentro[j - 1] = True

señal_dentro = float(var_predec[_dentro].sum())
señal_fuera  = float(var_predec[~_dentro].sum())

tabla_senal = pd.DataFrame({
    "j_fourier":        np.arange(1, J_GEN + 1),
    "phi":              PHIS,
    "var_empirica":     var_A,
    "var_predecible":   var_predec,
    "frac_senal_total": var_predec / max(señal_total, 1e-15),
    "dentro_del_truncamiento": _dentro,
})
tabla_senal.to_csv(PATHS["out_report"] / "62_senal_por_componente.csv", index=False)

display(tabla_senal.style.format({
    "phi": "{:+.2f}", "var_empirica": "{:.5f}", "var_predecible": "{:.5f}",
    "frac_senal_total": "{:.2%}"})
    .background_gradient(subset=["frac_senal_total"], cmap="Reds")
    .set_caption("Señal predecible por componente de Fourier del generador "
                 "(phi^2 * var) y si sobrevive al truncamiento"))

print(f"\nvarianza predecible TOTAL del proceso     : {señal_total:.6f}")
print(f"  retenida por el truncamiento a M={M_fpca}      : {señal_dentro:.6f}   "
      f"({señal_dentro / max(señal_total, 1e-15):.1%})")
print(f"  DESCARTADA por el truncamiento           : {señal_fuera:.6f}   "
      f"({señal_fuera / max(señal_total, 1e-15):.1%})")
print(f"\nvarianza TOTAL del proceso                : {var_A.sum():.6f}")
print(f"  retenida por el truncamiento             : "
      f"{var_A[_dentro].sum() / var_A.sum():.1%}")
print("\n   La asimetría entre estas dos cifras es el escenario completo: el "
      "truncamiento\n   puede retener casi toda la VARIANZA y a la vez "
      "descartar casi toda la SEÑAL.")

FRAC_SENAL_RETENIDA = float(señal_dentro / max(señal_total, 1e-15))

In [ ]:
# ── Veredicto de la sección, y de la corrida ────────────────────────────────
print("=" * 72)
if RETIENE:
    print("VEREDICTO — CASO NULO")
    print(f"""
  A M = {M_fpca} el truncamiento RETIENE la dirección informativa: el
  {FRAC_SENAL_RETENIDA:.1%} de la varianza predecible del proceso sobrevive. El
  Algoritmo 5 no somete a prueba nada en esta corrida, porque su tesis es que la
  dirección informativa es una de las que el truncamiento descarta.

  La corrida es válida y reportable como CONTROL: confirma que, cuando la
  reducción de dimensión no destruye la señal, el método recupera la dinámica
  —lo que se comprueba en el R2 de la componente con phi != 0 en §4—. Pero NO
  responde la pregunta de docs/03 Modelo.tex §03_06.

  Para responderla hace falta repetir la corrida con
      M <= {int(REPR_CFG.get('M_umbral_para_retener', M_fpca)) - 1}
  y un EXPERIMENT_ID distinto. Ver 15_01 §3.4.""")
else:
    print("VEREDICTO — EL ESCENARIO MUERDE")
    print(f"""
  A M = {M_fpca} el truncamiento DESCARTA la dirección informativa: sólo el
  {FRAC_SENAL_RETENIDA:.1%} de la varianza predecible del proceso sobrevive. Los
  scores retenidos son ruido blanco por construcción, de modo que el R2
  alcanzable es cero y la media incondicional es la predicción óptima.

  Insistir: esto NO es un fallo del PSBPM-FD. Alcanza por igual al FAR(1), al
  VAR sobre scores y a cualquier método sobre la misma representación. Lo que
  la corrida acota es el ALCANCE DE LA REDUCCIÓN DE DIMENSIÓN, y ésa es la
  forma en que hay que reportarla.""")
print("=" * 72)

## 10. Comparación con las líneas base

In [ ]:
_bl = PATHS["out_report"] / "30_baselines_test.csv"
rmse_psbp = met_df.loc["test", "RMSE"].mean()
print(f"RMSE promedio en scores — PSBP-FD (test): {rmse_psbp:.4f}\n")

if _bl.exists():
    baselines_df = pd.read_csv(_bl, index_col=0)
    display(baselines_df.style.format("{:.4f}", na_rep="—")
            .set_caption("Líneas base sobre el bloque de prueba (h=1)"))
    if "RMSE_scores_prom" in baselines_df.columns:
        for modelo, fila in baselines_df.iterrows():
            marca = "PSBP mejor" if rmse_psbp < fila["RMSE_scores_prom"] else "baseline mejor"
            print(f"  vs {modelo:24s} RMSE={fila['RMSE_scores_prom']:.4f}  → {marca}")
else:
    print("[AVISO] falta 30_baselines_test.csv — ejecuta 15_01 §6.")

print("\nLectura para el Escenario 5:")
if RETIENE:
    print("  Con la dirección informativa DENTRO del truncamiento, superar a la "
          "persistencia\n  en la componente con dinámica sí es exigible: hay "
          "señal que recoger. En las\n  demás componentes el empate con la "
          "media incondicional es el resultado correcto.")
else:
    print("  Con la dirección informativa FUERA del truncamiento, el empate con "
          "la media\n  incondicional es el resultado CORRECTO y no una derrota "
          "del modelo: no queda\n  señal que ningún método pueda recoger. Ésa "
          "es la medición del escenario.")
print("\nNota: sólo hay dos líneas base (media incondicional y persistencia). "
      "FAR(1),\nVAR sobre scores y ARIMA por score siguen pendientes en "
      "fit/baselines.py. En este\nescenario su ausencia pesa MENOS que en los "
      "demás, precisamente porque el\nresultado no discrimina entre "
      "especificaciones dinámicas.")

In [ ]:
# ── Resumen ejecutable del experimento ───────────────────────────────────────
resumen = {
    "experiment_id": EXPERIMENT_ID,
    "escenario_id": int(ESCENARIO_ID),
    "bloque_del_anexo": 2,
    "objetivo_evaluacion": OBJETIVO,
    "modo_residuo": MODO_RESIDUO,
    "nivel": NIVEL,
    "S_scores": int(S_total), "S_funcional": int(X_draws.shape[0]),
    # Representación: el objeto de estudio de este escenario
    "M_retenidas": int(M_fpca),
    "J_generador": int(J_GEN),
    "fourier_con_dinamica": int(J_ESTRELLA),
    "fpc_con_dinamica": int(REPR_CFG.get("fpc_con_dinamica", -1)),
    "retiene_direccion_informativa": bool(RETIENE),
    "es_caso_nulo": bool(RETIENE),
    "fraccion_senal_predecible_retenida": float(FRAC_SENAL_RETENIDA),
    "fraccion_varianza_retenida": float(var_A[_dentro].sum() / var_A.sum()),
    # Puntual
    "rmse_scores_train": float(met_df.loc["train", "RMSE"].mean()),
    "rmse_scores_test":  float(met_df.loc["test", "RMSE"].mean()),
    "r2_scores_test":    float(met_df.loc["test", "R2"].mean()),
    "crps_scores_test":  float(met_df.loc["test", "CRPS"].mean()),
    # Funcional
    "mise_train": float(fun_df.loc["train", "MISE"]),
    "mise_test":  float(fun_df.loc["test", "MISE"]),
    "mise_truncamiento_test": float(fun_df.loc["test", "MISE_truncamiento"]),
    "mise_bspline": float(_mise_bspl),
    "energy_test": float(fun_df.loc["test", "energy"]),
    "cobertura_puntual_test": float(fun_df.loc["test", f"Cob{int(NIVEL*100)}_puntual"]),
}
pd.Series(resumen).to_csv(PATHS["out_report"] / "65_resumen.csv", header=False)
for k, v in resumen.items():
    print(f"  {k:36s}: {v}")
print(f"\nFiguras y tablas en {PATHS['out_report']}")
print("\nAl reportar, encabezar SIEMPRE con la advertencia del Bloque 2: el "
      "resultado no\ndiscrimina entre especificaciones dinámicas; acota el "
      "alcance de la reducción\nde dimensión.")